# 🚀 DeepSeek-V4-Flash Model with Azure AI Inference 🧠

**DeepSeek-V4-Flash** is DeepSeek's low-latency, cost-efficient Mixture-of-Experts (MoE) model in the DeepSeek-V4 series, built for high-volume, real-time interactions. It has 284B total parameters with only 13B activated per token, and a 1M-token context window. It's available in Microsoft Foundry (preview).

In this notebook, you'll learn to:
1. **Initialize** the ChatCompletionsClient for Azure serverless endpoints
2. **Chat** with DeepSeek-V4-Flash using reasoning extraction
3. **Implement** a travel planning example with step-by-step reasoning
4. **Leverage** the 1M-token context window for complex scenarios

## Why DeepSeek-V4-Flash?
- **Low Latency**: Optimized for fast, high-volume real-time responses
- **Massive Context**: 1M-token window for detailed, long-document analysis
- **Efficient Architecture**: MoE with 13B active parameters from 284B total
- **Cost Efficient**: Designed for high-throughput serverless workloads

## 1. Setup & Authentication

Required packages:
- `azure-ai-inference`: For chat completions
- `python-dotenv`: For environment variables

.env file requirements:
```bash
AZURE_INFERENCE_ENDPOINT=<your-endpoint-url>
AZURE_INFERENCE_KEY=<your-api-key>
MODEL_NAME=DeepSeek-V4-Flash
```

In [ ]:
import os
import re
from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential
from pathlib import Path

# Load environment variables
notebook_path = Path().absolute()
parent_dir = notebook_path.parent.parent
load_dotenv(parent_dir / '.env')

# Load environment
load_dotenv()
endpoint = os.getenv("AZURE_INFERENCE_ENDPOINT")
key = os.getenv("AZURE_INFERENCE_KEY")
model_name = os.getenv("MODEL_NAME")
# DeepSeek-V4 models require a newer API version than the SDK default (2024-05-01-preview)
api_version = os.getenv("AZURE_INFERENCE_API_VERSION", "2024-12-01-preview")

# Initialize client
try:
    client = ChatCompletionsClient(
        endpoint=endpoint,
        credential=AzureKeyCredential(key),
        api_version=api_version
    )
    print("✅ Client initialized | Model:", model_name, "| API version:", api_version)
except Exception as e:
    print("❌ Initialization failed:", e)

## 2. Intelligent Travel Planning ✈️

Demonstrate DeepSeek-V4-Flash's reasoning capabilities for trip planning:

In [ ]:
def plan_trip_with_reasoning(query, show_thinking=False):
    """Get travel recommendations with reasoning extraction"""
    messages = [
        SystemMessage(content="You are a travel expert. Provide detailed plans with rationale."),
        UserMessage(content=f"{query} Include hidden gems and safety considerations.")
    ]
    
    response = client.complete(
        messages=messages,
        model=model_name,
        temperature=0.7,
        max_tokens=1024
    )
    
    content = response.choices[0].message.content
    
    # Extract reasoning if present
    if show_thinking:
        match = re.search(r"<think>(.*?)</think>(.*)", content, re.DOTALL)
        if match:
            return {"thinking": match.group(1).strip(), "answer": match.group(2).strip()}
    return content

# Example usage
query = "Plan a 5-day cultural trip to Kyoto in April"
result = plan_trip_with_reasoning(query, show_thinking=True)

print("🗺️ Query:", query)
if isinstance(result, dict):
    print("\n🧠 Thinking Process:", result["thinking"])
    print("\n📝 Final Answer:", result["answer"])
else:
    print("\n📝 Response:", result)

## 3. Technical Problem Solving 💻

Showcase coding/optimization capabilities:

In [ ]:
def solve_technical_problem(problem):
    """Solve complex technical problems with structured reasoning"""
    response = client.complete(
        messages=[
            UserMessage(content=f"{problem} Please reason step by step, and put your final answer within \boxed{{}}.")
        ],
        model=model_name,
        temperature=0.3,
        max_tokens=2048
    )
    
    return response.choices[0].message.content

# Database optimization example
problem = """How can I optimize a PostgreSQL database handling 10k transactions/second?
Consider indexing strategies, hardware requirements, and query optimization."""

print("🔧 Problem:", problem)
print("\n⚙️ Solution:", solve_technical_problem(problem))

## 4. Best Practices & Considerations

1. **Reasoning Handling**: Use regex to separate <think> content from final answers
2. **Safety**: Built-in content filtering - handle HttpResponseError for violations
3. **Performance**:
   - Max tokens: 4096
   - Rate limit: 200K tokens/minute
4. **Cost**: Pay-as-you-go with serverless deployment
5. **Streaming**: Implement response streaming for long completions

```python
# Streaming example
response = client.complete(..., stream=True)
for chunk in response:
    print(chunk.choices[0].delta.content or "", end="")
```

## 🎯 Key Takeaways
- Leverage the 1M-token context window for detailed, long-document analysis
- Extract reasoning steps for debugging/analysis
- Combine with Azure AI Content Safety for production
- Monitor token usage via response.usage

> Always validate model outputs for critical applications!